In [ ]:
import os



import json
import re
import gc
import torch
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM


print(" WEEK 4 RECOVERY PIPELINE (STRICT WORD LIMIT ENGINE)")


# 1. LOAD MODEL

MODEL_NAME = "ibm-granite/granite-3.3-8b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

if device == "cpu":
    model = model.to(device)



In [ ]:
RSF_PATH = ""

clusters = {}

with open(RSF_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) >= 3 and parts[0] == "contain":
            cluster = parts[1]
            classname = parts[2]

            clusters.setdefault(cluster, [])
            clusters[cluster].append(classname)

for k in clusters:
    clusters[k] = sorted(list(set(clusters[k])))

print("Clusters found:", len(clusters))


# 3. CORE LLM QUERY MECHANISM

def query_llm(prompt, tokens_limit=400):
    messages = [
        {
            "role": "system",
            "content": "You are a concise software architect. You summarize technical systems in short paragraphs. You must finish your sentences completely."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=tokens_limit,  # Wide token ceiling ensures sentences complete fully
            temperature=0.1,
            do_sample=False
        )

    result = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(result, skip_special_tokens=True).strip()

# 4. SUMMARIZE ONE CLUSTER WITH AUTO-RECOMPRESSION

def summarize_cluster(cluster_id, files):
    # Apply file-level mapping pre-processing step
    clean_files = sorted(list(set([f.split(".")[-1].split("$")[0] for f in files])))
    class_text = ", ".join(clean_files[:100])

    prompt = f"""Analyze these architectural components from Apache Hadoop MapReduce:
Cluster: {cluster_id}
Classes: {class_text}

Provide exactly:
Title: [A short 3-to-5 word functional module title]
Description: [A single continuous summary paragraph that explicitly mentions major components, interactions, technologies used, scalability, performance, and fault tolerance. Keep this under 130 words.]
"""

    output = query_llm(prompt, tokens_limit=400)

    # Parse description text to run length diagnostics
    desc_match = re.search(r"Description:\s*(.*)", output, re.DOTALL | re.IGNORECASE)
    description = desc_match.group(1).strip() if desc_match else output

    # 🔁 LENGTH CONSTRAINT POST-PROCESSOR ENGINE
    # If Granite breaches the word limit, it loops right here to condense the text safely.
    word_count = len(description.split())
    if word_count > 140:
        print(f"      Word limit breached ({word_count} words). Compressing...")
        compression_prompt = f"""Rewrite this software description into a single continuous paragraph that is strictly under 120 words. You must keep the core content intact but make sentences extremely concise. Finish your final sentence completely.

        Text to compress:
        {description}
        """
        compressed_desc = query_llm(compression_prompt, tokens_limit=250)

        # Strip structural cleanups out of the final compression loop result
        compressed_desc = re.sub(r"^(Description:)\s*", "", compressed_desc, flags=re.IGNORECASE)
        output = f"Title: {output.split('Description:')[0].replace('Title:', '').strip()}\nDescription: {compressed_desc.strip()}"

    return output

# 5. PROCESS ALL CLUSTERS

results = []
total = len(clusters)

for idx, (cluster_id, files) in enumerate(clusters.items(), start=1):
    print(f"[{idx}/{total}] Processing {cluster_id}")

    output = summarize_cluster(cluster_id, files)

    title_match = re.search(r"Title:\s*(.*?)\n", output, re.IGNORECASE)
    desc_match = re.search(r"Description:\s*(.*)", output, re.DOTALL | re.IGNORECASE)

    title = title_match.group(1).strip() if title_match else f"MapReduce Subsystem - {cluster_id}"
    description = desc_match.group(1).strip() if desc_match else output

    # Sanitize markdown artifacts for spreadsheet integration
    title = title.replace('**', '').replace('`', '').strip()
    description = description.replace('**', '').replace('`', '').strip()

    final_files_list = sorted(list(set([f.split(".")[-1].split("$")[0] for f in files])))
    final_word_count = len(description.split())

    print(f"       Summary written successfully ({final_word_count} words).")

    results.append({
        "cluster_ID": cluster_id,
        "files": ", ".join(final_files_list),
        "title": title,
        "description": description
    })

    # Continuous disk saving to prevent loss
    pd.DataFrame(results).to_csv("ARC_OUTPUT.csv", index=False)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 6. FINAL SHEET VERIFICATION EXPORT
df = pd.DataFrame(results)
df = df[["cluster_ID", "files", "title", "description"]]
df.to_csv("ARC_OUTPUT.csv", index=False)

print("\n ALL TASKS COMPLETE!")
print("Saved perfect professor-compliant file: ARC_OUTPUT.csv")

